# Modular API

**Objective:** Connect HTTP, service, repository, and infrastructure as replaceable layers.

## Simple version

In [ ]:
layers = ["HTTP", "service", "repository", "database"]
print(" -> ".join(layers))

## Polished version

In [ ]:
from typing import Optional, Protocol

import httpx
from fastapi import APIRouter, FastAPI, HTTPException, status
from pydantic import BaseModel, Field


class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)


class TaskResponse(BaseModel):
    id: int
    title: str


class TaskRepository(Protocol):
    async def get(self, task_id: int) -> Optional[TaskResponse]: ...
    async def add(self, body: TaskCreate) -> TaskResponse: ...


class MemoryTaskRepository:
    def __init__(self) -> None:
        self.tasks: dict[int, TaskResponse] = {}

    async def get(self, task_id: int) -> Optional[TaskResponse]:
        return self.tasks.get(task_id)

    async def add(self, body: TaskCreate) -> TaskResponse:
        task = TaskResponse(id=len(self.tasks) + 1, title=body.title)
        self.tasks[task.id] = task
        return task


class TaskService:
    def __init__(self, tasks: TaskRepository) -> None:
        self.tasks = tasks

    async def create(self, body: TaskCreate) -> TaskResponse:
        return await self.tasks.add(body)

    async def get(self, task_id: int) -> TaskResponse:
        task = await self.tasks.get(task_id)
        if task is None:
            raise KeyError(task_id)
        return task


def create_app(service: TaskService) -> FastAPI:
    router = APIRouter(prefix="/tasks")

    @router.post("", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
    async def create_task(body: TaskCreate) -> TaskResponse:
        return await service.create(body)

    @router.get("/{task_id}", response_model=TaskResponse)
    async def get_task(task_id: int) -> TaskResponse:
        try:
            return await service.get(task_id)
        except KeyError as error:
            raise HTTPException(404, "Task not found") from error

    app = FastAPI()
    app.include_router(router)
    return app


repository = MemoryTaskRepository()
app = create_app(TaskService(repository))
transport = httpx.ASGITransport(app=app)

async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    created = await client.post("/tasks", json={"title": "Wire modules"})
    fetched = await client.get(f"/tasks/{created.json()['id']}")

print(created.status_code, fetched.json())